# 🎈 Balloon Detection — YOLOv11, YOLOv26

## 0. Install dependencies

In [ ]:
!pip install ultralytics mlflow optuna optuna-integration[mlflow] plotly nbformat

## 1. Configuration

In [ ]:
import os
import random
import numpy as np
import torch
import mlflow
import optuna
from ultralytics import YOLO, settings
from pathlib import Path

# ─────────────────────────────────────────────
# 📁 PATHS — change to your actual paths
# ─────────────────────────────────────────────
DATA_YAML  = "dataset/data.yaml"
RUNS_DIR   = "runs/balloon_detection"

# ─────────────────────────────────────────────
# 📊 MLFLOW
# Local: "runs/mlflow"  |  Remote: "http://your-server-ip:5000"
# ─────────────────────────────────────────────
MLFLOW_TRACKING_URI    = "runs/mlflow"
MLFLOW_EXPERIMENT_NAME = "balloon_detection"

# ─────────────────────────────────────────────
# 🔬 STUDY CONFIG
# ─────────────────────────────────────────────
MODELS_TO_SEARCH   = ["yolo11s.pt", "yolo26s.pt"]
N_TRIALS_PER_MODEL = 100 
EPOCHS             = 30
IMGSZ              = 640
METRIC_TO_OPTIMIZE = "metrics/mAP50(B)"

# ─────────────────────────────────────────────
# 🌱 REPRODUCIBILITY
# ─────────────────────────────────────────────
SEED            = 42
DATASET_VERSION = "v1.0"

# ─────────────────────────────────────────────
# ⚙️ ULTRALYTICS — отключаем автоинтеграцию
# ─────────────────────────────────────────────
settings.update({"mlflow": False})

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"✅ MLflow tracking URI : {MLFLOW_TRACKING_URI}")
print(f"✅ Experiment name     : {MLFLOW_EXPERIMENT_NAME}")
print(f"✅ Models to search    : {MODELS_TO_SEARCH}")
print(f"✅ Trials per model    : {N_TRIALS_PER_MODEL}")
print(f"✅ Seed                : {SEED}")
print(f"✅ Dataset version     : {DATASET_VERSION}")

## 2. Reproducibility — set global seed

In [ ]:
def set_seed(seed: int = 42) -> None:
    """Fix all random sources for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic=True #устраняет недетерминизм в CUDA-ядрах
    # benchmark=False #отключает авто-подбор алгоритма свёртки
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"✅ Global seed set to {seed}")

set_seed(SEED)

## 3. Dataset validation

In [ ]:
import yaml

assert Path(DATA_YAML).exists(), f"❌ data.yaml not found: {DATA_YAML}"

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

required_keys = {"train", "val", "nc", "names"}
missing = required_keys - set(data_cfg.keys())
assert not missing, f"❌ data.yaml is missing keys: {missing}"

print("data.yaml contents:")
for k, v in data_cfg.items():
    print(f"  {k}: {v}")

for split in ("train", "val", "test"):
    split_path = data_cfg.get(split)
    if split_path is None:
        print(f"⚠️  '{split}' key not found in data.yaml — skipping")
        continue
    p = Path(split_path)
    if not p.exists():
        print(f"⚠️  {split} dir not found: {p}")
        continue
    n_images = len(list(p.glob("*.jpg")) + list(p.glob("*.png")))
    assert n_images > 0, f"❌ No images found in {p}"
    print(f"✅ {split:5s}: {n_images} images at {p}")

print(f"\n✅ Dataset version : {DATASET_VERSION}")
print(f"   Classes        : {data_cfg['nc']} → {data_cfg['names']}")

## 4. Hyperparameter search space

In [ ]:
def suggest_hyperparams(trial: optuna.Trial) -> dict:
    """
    Search space tuned for single-class balloon detection with YOLOv11.
    """
    return {
        # ── Learning rate ────────────────────────────────────────────
        "lr0": trial.suggest_float("lr0", 1e-5, 1e-1, log=True),
        "lrf": trial.suggest_float("lrf", 0.01, 0.3),

        # ── Regularisation ───────────────────────────────────────────
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        "momentum":     trial.suggest_float("momentum", 0.80, 0.99),

        # ── Batch & optimizer ────────────────────────────────────────
        "batch":     trial.suggest_categorical("batch",     [16, 32, 64]),
        "optimizer": trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW"]),

        # ── Warmup ───────────────────────────────────────────────────
        "warmup_epochs": trial.suggest_categorical("warmup_epochs", [2, 3, 5]),

        # ── Color augmentation ───────────────────────────────────────
        # hsv_h расширен до 0.15: шарики бывают любого цвета
        "hsv_h": trial.suggest_float("hsv_h", 0.0, 0.15),
        "hsv_s": trial.suggest_float("hsv_s", 0.0, 0.90),
        "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.90),

        # ── Geometric augmentation ───────────────────────────────────
        "degrees":   trial.suggest_float("degrees",   0.0, 10.0),
        "scale":     trial.suggest_float("scale",     0.3,  0.7),
        "translate": trial.suggest_float("translate", 0.0,  0.2),

        # ── Flip augmentation ────────────────────────────────────────
        "flipud": trial.suggest_float("flipud", 0.0, 0.3),
        "fliplr": trial.suggest_float("fliplr", 0.0, 0.5),

        # ── Mosaic / Mixup ───────────────────────────────────────────
        "mosaic": trial.suggest_float("mosaic", 0.5, 1.0),
        "mixup":  trial.suggest_float("mixup",  0.0, 0.1),
    }

print("✅ Search space defined — 18 hyperparameters")

## 5. Objective function — one Optuna trial = one YOLO training run

In [ ]:
def make_objective(model_checkpoint: str):
    """
    Factory: возвращает objective-функцию, привязанную к конкретному чекпоинту.
    trial передаётся через closure — никакого threading.local.
    """

    def objective(trial: optuna.Trial) -> float:
        params = suggest_hyperparams(trial)

        run_name = (
            f"{Path(model_checkpoint).stem}"
            f"__trial{trial.number}"
            f"__lr{params['lr0']:.5f}"
            f"__bs{params['batch']}"
            f"__{params['optimizer']}"
        )

        try:
            mlflow.end_run()
        except Exception:
            pass

        pruned_at_epoch_holder = [-1]

        with mlflow.start_run(run_name=run_name):
            mlflow.log_param("model",           model_checkpoint)
            mlflow.log_param("epochs",          EPOCHS)
            mlflow.log_param("imgsz",           IMGSZ)
            mlflow.log_param("seed",            SEED)
            mlflow.log_param("dataset_version", DATASET_VERSION)
            mlflow.log_param("trial",           trial.number)
            for k, v in params.items():
                mlflow.log_param(k, v)

            try:
                model = YOLO(model_checkpoint)

                def on_fit_epoch_end(trainer):
                    epoch         = trainer.epoch
                    current_map50 = trainer.metrics.get("metrics/mAP50(B)", 0.0)
                    mlflow.log_metric("epoch_mAP50", current_map50, step=epoch)
                    trial.report(current_map50, step=epoch)
                    if trial.should_prune():
                        pruned_at_epoch_holder[0] = epoch  # сохраняем эпоху среза
                        raise optuna.TrialPruned()

                model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

                results = model.train(
                    data     = DATA_YAML,
                    epochs   = EPOCHS,
                    imgsz    = IMGSZ,
                    seed     = SEED,
                    project  = RUNS_DIR,
                    name     = run_name,
                    exist_ok = True,
                    verbose  = False,
                    **params
                )

                metrics = results.results_dict
                map50   = metrics.get(METRIC_TO_OPTIMIZE, 0.0)

                mlflow.log_metric("mAP50",        metrics.get("metrics/mAP50(B)",    0.0))
                mlflow.log_metric("mAP50_95",     metrics.get("metrics/mAP50-95(B)", 0.0))
                mlflow.log_metric("precision",    metrics.get("metrics/precision(B)", 0.0))
                mlflow.log_metric("recall",       metrics.get("metrics/recall(B)",   0.0))
                mlflow.log_metric("box_loss",     metrics.get("val/box_loss",        0.0))
                mlflow.log_metric("cls_loss",     metrics.get("val/cls_loss",        0.0))
                mlflow.log_metric("optuna_value", map50)
                mlflow.set_tag("status", "completed")

                best_weights = Path(RUNS_DIR) / run_name / "weights" / "best.pt"
                if best_weights.exists():
                    mlflow.log_artifact(str(best_weights), artifact_path="weights")

                print(f"  ✅ Trial {trial.number} | {run_name} | mAP50={map50:.4f}")
                return map50

            except optuna.TrialPruned:
                epoch_pruned = pruned_at_epoch_holder[0]
                mlflow.set_tag("status",          "pruned")
                mlflow.set_tag("pruned_at_epoch", str(epoch_pruned)) 
                print(f"  ✂️  Trial {trial.number} pruned at epoch {epoch_pruned}")
                raise

            except Exception as e:
                mlflow.set_tag("status", "failed")
                mlflow.set_tag("error",  str(e))
                print(f"  ❌ Trial {trial.number} failed: {e}")
                return 0.0

    return objective


print("✅ Objective function с межэпочным pruning определена")

## 6. Run Optuna search for each model

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_studies = {}

for checkpoint in MODELS_TO_SEARCH:
    model_name = Path(checkpoint).stem
    study_name = f"{MLFLOW_EXPERIMENT_NAME}__{model_name}"

    print(f"\n{'='*60}")
    print(f"🔍 Starting search for: {checkpoint}")
    print(f"   Study name : {study_name}")
    print(f"   Trials     : {N_TRIALS_PER_MODEL}")
    print(f"{'='*60}")

    sampler = optuna.samplers.TPESampler(seed=SEED)

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,  
        n_warmup_steps=10,
    )

    study = optuna.create_study(
        study_name     = study_name,
        direction      = "maximize",
        sampler        = sampler,
        pruner         = pruner,
        storage        = f"sqlite:///optuna_{model_name}.db",
        load_if_exists = True,
    )

    study.optimize(
        make_objective(checkpoint),
        n_trials          = N_TRIALS_PER_MODEL,
        show_progress_bar = True,
    )

    all_studies[model_name] = study

    print(f"\n🏆 Best trial for {model_name}:")
    print(f"   mAP50  : {study.best_value:.4f}")
    print(f"   Params : {study.best_params}")

print("\n✅ All studies complete!")

## Загрузить уже завершённые исследования

In [ ]:
all_studies = {}

for checkpoint in MODELS_TO_SEARCH:
    model_name = Path(checkpoint).stem
    study_name = f"{MLFLOW_EXPERIMENT_NAME}__{model_name}"
    db_path    = f"sqlite:///optuna_{model_name}.db"
    study = optuna.load_study(study_name=study_name, storage=db_path)
    all_studies[model_name] = study
    print(f"✅ Loaded: {model_name} — {len(study.trials)} trials, best mAP50={study.best_value:.4f}")

## 7. Results summary — compare all models

In [ ]:
import pandas as pd

rows = []
for model_name, study in all_studies.items():
    best = study.best_trial
    row  = {"model": model_name, "mAP50": best.value, "trial_number": best.number}
    row.update(best.params)
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values("mAP50", ascending=False)
summary_df = summary_df.reset_index(drop=True)
summary_df.index += 1

print("📊 Best hyperparameters per model (ranked by mAP50):")
print(summary_df.to_string())

In [ ]:
for model_name, study in all_studies.items():
    df = study.trials_dataframe()
    print(f"\n📋 All trials for {model_name} (sorted by mAP50):")
    cols      = ["number", "value", "state"] + [c for c in df.columns if c.startswith("params_")]
    available = [c for c in cols if c in df.columns]
    print(df[available].sort_values("value", ascending=False).to_string(index=False))

## 8. Optuna visualizations

In [ ]:
from optuna import visualization

for model_name, study in all_studies.items():
    print(f"\n📈 Optimization history — {model_name}")
    fig = visualization.plot_optimization_history(study)
    fig.update_layout(title=f"Optimization History — {model_name}")
    fig.show()

    print(f"\n🔑 Hyperparameter importance — {model_name}")
    fig = visualization.plot_param_importances(study)
    fig.update_layout(title=f"Hyperparameter Importance — {model_name}")
    fig.show()

    print(f"\n🌊 Parallel coordinate plot — {model_name}")
    fig = visualization.plot_parallel_coordinate(study)
    fig.update_layout(title=f"Parallel Coordinates — {model_name}")
    fig.show()

## 9. Query MLflow — load all experiment runs programmatically

In [ ]:
from mlflow.tracking import MlflowClient

client     = MlflowClient(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

runs_df = mlflow.search_runs(
    experiment_ids = [experiment.experiment_id],
    order_by       = ["metrics.mAP50 DESC"],
)

metric_cols = [c for c in runs_df.columns if c.startswith("metrics.")]
param_cols  = [c for c in runs_df.columns if c.startswith("params.")]
key_cols    = ["tags.mlflow.runName", "status"] + metric_cols + param_cols
available   = [c for c in key_cols if c in runs_df.columns]

print(f"Total runs found: {len(runs_df)}")
print("\n📊 Top 10 runs by mAP50:")
runs_df[available].head(10)

In [ ]:
best_run = runs_df.iloc[0]

print("🏆 BEST RUN OVERALL")
print(f"  Run name   : {best_run.get('tags.mlflow.runName', best_run['run_id'])}")
print(f"  Model      : {best_run.get('params.model', 'N/A')}")
print(f"  Dataset v  : {best_run.get('params.dataset_version', 'N/A')}")
print(f"  Seed       : {best_run.get('params.seed', 'N/A')}")
print(f"  mAP50      : {best_run.get('metrics.mAP50', 0):.4f}")
print(f"  mAP50-95   : {best_run.get('metrics.mAP50_95', 0):.4f}")
print(f"  Precision  : {best_run.get('metrics.precision', 0):.4f}")
print(f"  Recall     : {best_run.get('metrics.recall', 0):.4f}")
print(f"  lr0        : {best_run.get('params.lr0', 'N/A')}")
print(f"  batch      : {best_run.get('params.batch', 'N/A')}")
print(f"  optimizer  : {best_run.get('params.optimizer', 'N/A')}")

BEST_RUN_ID = best_run["run_id"]

## 10. Download best weights from MLflow & run final evaluation

In [ ]:
artifacts = client.list_artifacts(BEST_RUN_ID, "weights")

if artifacts:
    local_dir = mlflow.artifacts.download_artifacts(
        run_id        = BEST_RUN_ID,
        artifact_path = "weights",
        dst_path      = "./best_model"
    )
    best_weights_path = str(Path(local_dir) / "best.pt")
    print(f"✅ Best weights downloaded to: {best_weights_path}")
else:
    run_name          = best_run.get("tags.mlflow.runName", "")
    best_weights_path = str(Path(RUNS_DIR) / run_name / "weights" / "best.pt")
    print(f"⚠️  Artifacts not in MLflow, using local path: {best_weights_path}")

print(f"   Weights file exists: {Path(best_weights_path).exists()}")

In [ ]:
best_model = YOLO(best_weights_path)

test_results = best_model.val(
    data    = DATA_YAML,
    split   = "test",
    imgsz   = IMGSZ,
    verbose = True,
)

with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_metric("test/mAP50",     test_results.results_dict.get("metrics/mAP50(B)",    0))
    mlflow.log_metric("test/mAP50_95",  test_results.results_dict.get("metrics/mAP50-95(B)", 0))
    mlflow.log_metric("test/precision", test_results.results_dict.get("metrics/precision(B)", 0))
    mlflow.log_metric("test/recall",    test_results.results_dict.get("metrics/recall(B)",   0))
    mlflow.set_tag("stage", "final_test")

print("\n🎯 Final test set metrics logged to MLflow run:", BEST_RUN_ID)

## 11. Run inference on sample images

In [ ]:
import glob
from IPython.display import Image as IPImage, display

test_images = glob.glob("dataset/test/images/*.jpg")[:4]

if not test_images:
    print("⚠️  No test images found at dataset/test/images/*.jpg")
else:
    results = best_model.predict(
        source   = test_images,
        imgsz    = IMGSZ,
        conf     = 0.25,
        save     = True,
        project  = "runs/predict",
        name     = "balloon_best_model",
        exist_ok = True,
    )
    pred_images = glob.glob("runs/predict/balloon_best_model/*.jpg")
    for img_path in pred_images[:4]:
        display(IPImage(img_path, width=600))

## 12. Launch MLflow UI

Run the cell below to get the command to open MLflow UI.

> **Remote access**: if training on a server:
> ```bash
> ssh -L 5000:localhost:5000 user@your-server-ip
> ```
> Then open `http://localhost:5000`.

In [ ]:
# print("To open MLflow UI, run in terminal:")
# print(f"\n  mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI} --port 5000\n")
# print("Then open http://127.0.0.1:5000 in your browser.")
# print("\nIn the UI you can:")
# print("  • Select all runs → click 'Compare' for side-by-side table")
# print("  • Use 'Chart' view to plot mAP50 vs lr0, batch, optimizer...")
# print("  • Download best.pt from the Artifacts tab of any run")

## 13. Final full training with best hyperparameters

In [ ]:
import time

best_model_name = max(all_studies, key=lambda m: all_studies[m].best_value)
best_trial      = all_studies[best_model_name].best_trial
final_params    = dict(best_trial.params) 

RUN_NAME = f"FINAL__{best_model_name}__{int(time.time())}"

print(f"🏆 Модель    : {best_model_name}")
print(f"   Trial №  : {best_trial.number}")
print(f"   mAP50    : {best_trial.value:.4f}")
print(f"   Params   : {final_params}")
print(f"   Run name : {RUN_NAME}")

try:
    mlflow.end_run()
except Exception:
    pass

set_seed(SEED)  # сбрасываем seed перед финальным обучением
final_model = YOLO(f"{best_model_name}.pt")

with mlflow.start_run(run_name=RUN_NAME) as run:

    mlflow.log_param("model",           best_model_name)
    mlflow.log_param("source_trial",    best_trial.number)
    mlflow.log_param("source_mAP50",    round(best_trial.value, 4))
    mlflow.log_param("epochs",          150)
    mlflow.log_param("patience",        30)
    mlflow.log_param("seed",            SEED)
    mlflow.log_param("dataset_version", DATASET_VERSION)
    for k, v in final_params.items():
        mlflow.log_param(k, v)

    def on_fit_epoch_end(trainer):
        epoch    = trainer.epoch
        map50    = trainer.metrics.get("metrics/mAP50(B)", 0.0)
        box_loss = trainer.metrics.get("val/box_loss",     0.0)
        cls_loss = trainer.metrics.get("val/cls_loss",     0.0)
        mlflow.log_metric("epoch_mAP50",    map50,    step=epoch)
        mlflow.log_metric("epoch_box_loss", box_loss, step=epoch)
        mlflow.log_metric("epoch_cls_loss", cls_loss, step=epoch)

    final_model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

    results = final_model.train(
        data     = DATA_YAML,
        epochs   = 150,
        patience = 30,
        imgsz    = IMGSZ,
        seed     = SEED,    
        project  = RUNS_DIR,
        name     = RUN_NAME,
        exist_ok = False,   
        verbose  = True,
        **final_params     
    )

    metrics = results.results_dict
    mlflow.log_metric("final/mAP50",      metrics.get("metrics/mAP50(B)",    0))
    mlflow.log_metric("final/mAP50_95",   metrics.get("metrics/mAP50-95(B)", 0))
    mlflow.log_metric("final/precision",  metrics.get("metrics/precision(B)", 0))
    mlflow.log_metric("final/recall",     metrics.get("metrics/recall(B)",   0))
    mlflow.log_metric("final/box_loss",   metrics.get("val/box_loss",        0))
    mlflow.log_metric("final/cls_loss",   metrics.get("val/cls_loss",        0))
    mlflow.set_tag("stage", "final_training")

    weights_dir  = Path(RUNS_DIR) / RUN_NAME / "weights"
    best_weights = weights_dir / "best.pt"
    last_weights = weights_dir / "last.pt"

    for pt_file in [best_weights, last_weights]:
        if pt_file.exists():
            mlflow.log_artifact(str(pt_file), artifact_path="weights")
            print(f"✅ Залогирован: {pt_file.name}")

    FINAL_RUN_ID  = run.info.run_id
    final_weights = best_weights if best_weights.exists() else last_weights

    print(f"\n🎯 Обучение завершено!")
    print(f"   Веса        : {final_weights}")
    print(f"   MLflow run  : {FINAL_RUN_ID}")
    print(f"   mAP50       : {metrics.get('metrics/mAP50(B)',    0):.4f}")
    print(f"   mAP50-95    : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"   Precision   : {metrics.get('metrics/precision(B)', 0):.4f}")
    print(f"   Recall      : {metrics.get('metrics/recall(B)',   0):.4f}")

## 14. Export best model to ONNX (optional)

In [ ]:
# Uncomment to export the best model to ONNX format

# onnx_path = final_model.export(format="onnx", imgsz=IMGSZ)
# print(f"✅ Exported to ONNX: {onnx_path}")
#
# with mlflow.start_run(run_id=FINAL_RUN_ID):
#     mlflow.log_artifact(onnx_path, artifact_path="export")
#     print("✅ ONNX artifact logged to MLflow")